In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

In [ ]:
from calibration_settings_patch import DATASET, SETTINGS, OUTPUT_DIR, rows
print(f"Dataset: {DATASET}")
discharge_directory = OUTPUT_DIR

points_of_interest_file_name = Path.joinpath(discharge_directory, "points_of_interest.json")
battery_state_file_name = Path.joinpath(discharge_directory, "battery_state.csv")
compiled_file_name = Path.joinpath(discharge_directory, "final_cleaned_compiled (with SoC).csv")

output_file_name = Path.joinpath(discharge_directory, "battery_constants.json")

Dataset: LiFePO4


In [3]:
compiled_df = pd.read_csv(compiled_file_name)
print(compiled_df.head())

   Translated Time  Translated Time (minutes)  \
0            0.000                   0.000000   
1            0.942                   0.000016   
2            1.880                   0.000031   
3            2.819                   0.000047   
4            3.757                   0.000063   

   ADS Reading 1 (Smoothed and Offset)  Calibrated Current 1  \
0                              32755.0             -0.030601   
1                              32746.0             -0.051791   
2                              32737.0             -0.072985   
3                              32755.0             -0.030601   
4                              32737.0             -0.072985   

   ADS Reading 2 (Smoothed and Offset)  Calibrated Current 2  \
0                              32730.0             -0.089471   
1                              32722.5             -0.107137   
2                              32722.0             -0.108315   
3                              32722.0             -0.108315   


In [4]:
sensor_draw_estimate_Ah = 0.081
sensor_draw_estimate_coulombs = sensor_draw_estimate_Ah * 3600

total_charge = compiled_df['Charge Change'].sum() - sensor_draw_estimate_coulombs
total_Ah = (total_charge / 3600)

print(f"Total Charge: {total_charge:.4f} Coulombs")
print(f"Total Ah: {total_Ah:.4f} Ah")

Total Charge: 133244.1810 Coulombs
Total Ah: 37.0123 Ah


In [5]:
points_of_interest = json.load(open(points_of_interest_file_name, 'r'))
before_close = points_of_interest['before_close']
before_open = points_of_interest['before_open']

In [ ]:
STEP_BACK = 10
SAMPLING_RANGE = 50

sd_voltage_array = []

for index in before_close['index'][:-1]: # Exclude the last point which have the highest noise from shutting down
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].mean()
    sd_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].std()
    sd_voltage_array.append(sd_voltage)
    print(f"Mean Voltage: {mean_voltage:.4f} V, SD Voltage: {sd_voltage:.4f} V")

sigma_v = np.nanmean(sd_voltage_array)
print(f"Mean Standard Deviations: {sigma_v:.4f} V")

Mean Voltage: 52.8269 V, SD Voltage: 0.0008 V
Mean Voltage: 52.6349 V, SD Voltage: 0.0008 V
Mean Voltage: 52.3253 V, SD Voltage: 0.0008 V
Mean Voltage: 52.2212 V, SD Voltage: 0.0007 V
Mean Voltage: 52.1880 V, SD Voltage: 0.0014 V
Mean Voltage: 51.9031 V, SD Voltage: 0.0007 V
Mean Standard Deviations: 0.0009 V


In [7]:
# During circuit open, where current = 0, almost no noise.
# Need to use a realisitic point where current is flowing to estimate the noise in current measurement.

STEP_FORWARD = 10
SAMPLING_RANGE = 500

sd_zero_current_array = []

for index in before_open['index']:
    start_index = min(len(compiled_df) - 1, index + STEP_FORWARD)
    end_index = min(len(compiled_df) - 1, start_index + SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    sd_zero_current_array.append(sd_current)
    
    print(f"Mean Current: {mean_current:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_zero_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 25.4285 A, SD Current: 0.0212 A
Mean Current: 26.0530 A, SD Current: 0.0329 A
Mean Current: 25.4212 A, SD Current: 0.0269 A
Mean Current: 25.0319 A, SD Current: 0.0315 A
Mean Current: 24.9704 A, SD Current: 0.0336 A
Mean Current: 25.0509 A, SD Current: 0.0310 A
Mean Current: 24.6887 A, SD Current: 0.0360 A
Mean Standard Deviations: 0.0304 A


In [8]:
STEP_BACK = 10
SAMPLING_RANGEs = [100, 200, 300, 400]

for sampling_range in SAMPLING_RANGEs:
    sd_current_array = []

    for index in before_open['index']:
        end_index = max(0, index - STEP_BACK)
        start_index = max(0, end_index - sampling_range)
        mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
        mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
        
        sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
        sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
        
        mean_current = (mean_current_1 + mean_current_2) / 2
        sd_current = (sd_current_1 + sd_current_2) / 2
        
        sd_current_array.append(sd_current)
        #print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current_1:.4f} A")
    print(f"Mean Standard Deviations ({sampling_range}) samples: {np.mean(sd_current_array):.4f} A")

Mean Standard Deviations (100) samples: 0.0286 A
Mean Standard Deviations (200) samples: 0.0285 A
Mean Standard Deviations (300) samples: 0.0290 A
Mean Standard Deviations (400) samples: 0.0301 A


In [9]:
STEP_BACK = 10
SAMPLING_RANGE = 200

sd_current_array = []

for index in before_open['index']:
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    
    sd_current_array.append(sd_current)
    print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 25.4020 A, SD Current: 0.0213 A
Mean Current: 25.9713 A, SD Current: 0.0275 A
Mean Current: 25.4163 A, SD Current: 0.0257 A
Mean Current: 25.0247 A, SD Current: 0.0307 A
Mean Current: 24.9685 A, SD Current: 0.0325 A
Mean Current: 25.0114 A, SD Current: 0.0302 A
Mean Current: 24.7030 A, SD Current: 0.0314 A
Mean Standard Deviations: 0.0285 A


In [10]:
sigma_kcl = np.sqrt(4* sigma_i**2)
print(f"Sigma KCL: {sigma_kcl:.4f} A")

Sigma KCL: 0.0570 A


In [11]:
battery_state = pd.read_csv(battery_state_file_name)
soc_factor = len(str(battery_state['SoC'].iloc[1]).split('.')[1])

print(soc_factor)

3


In [12]:
battery_constant = {
    'Q_total': total_Ah,
    'sigma_v': sigma_v,
    'sigma_i': sigma_i,
    'sigma_kcl': sigma_kcl,
    'interval_factor': soc_factor
}

print(f"Battery Constants: {json.dumps(battery_constant, indent=4)}")

Battery Constants: {
    "Q_total": 37.01227251145249,
    "sigma_v": 0.0008513771086966949,
    "sigma_i": 0.028479002137423107,
    "sigma_kcl": 0.056958004274846215,
    "interval_factor": 3
}


In [13]:
json.dump(battery_constant, open(output_file_name, 'w'), indent=4)
print(f"Battery constants saved to {output_file_name}")

Battery constants saved to C:\Users\Kor\Documents\GitHub\solar_proa_network_server\final_report\Calibration Code\LiFePO4 Patch Output\battery_constants.json
